In [1]:
from pathlib import Path

import numpy as np
from numba import njit
from scipy import stats
from tqdm import tqdm

import jitr

In [2]:
import matplotlib

matplotlib.use("pgf")
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.lines import Line2D

colors = [
    "#9467bd",
    "#2ca02c",
    "#d62728",
    "#8c564b",
    "#bcbd22",
]


rcParams["legend.fontsize"] = 14
rcParams["font.size"] = 14
rcParams["font.weight"] = "normal"
rcParams["xtick.labelsize"] = 14.0
rcParams["ytick.labelsize"] = 14.0
rcParams["lines.linewidth"] = 2.0
rcParams["xtick.major.pad"] = "10"
rcParams["ytick.major.pad"] = "10"

In [3]:
#  elastic reaction
target = (40, 20)
proton = (1, 1)
neutron = (1, 0)
projectile = neutron
Elab = 21
a = 5 * np.pi
sgrid = np.linspace(0.1, a, 100)
# for plotting differential xs
angles = np.linspace(0.1, np.pi, 100)

In [4]:
from jitr.optical_potentials.potential_forms import (
    coulomb_charged_sphere,
    thomas_safe,
    woods_saxon_safe,
)


def interaction_central(r, V, W, R, a, Zz, Rc):
    nucl = -(V + 1j * W) * woods_saxon_safe(r, R, a)
    coul = coulomb_charged_sphere(r, Zz, Rc)
    return nucl + coul


def interaction_spin_orbit(r, Vso, Wso, Rso, aso):
    return 2 * (Vso + 1j * Wso) * thomas_safe(r, Rso, aso)

In [5]:
solver = []
core_solver = jitr.rmatrix.Solver(50)
reaction = jitr.reactions.ElasticReaction(
    target=target,
    projectile=projectile,
)


# get kinematics and parameters for this experiment
kinematics = reaction.kinematics(Elab)

channel_radius_fm = a / kinematics.k

solver = jitr.xs.elastic.DifferentialWorkspace.build_from_system(
    reaction=reaction,
    kinematics=kinematics,
    channel_radius_fm=channel_radius_fm,
    solver=core_solver,
    lmax=50,
    angles=angles,
)

In [6]:
R = 1.2 * target[0] ** (1.0 / 3)
a = 0.7
Zz = target[1] * projectile[1]
V, W = 50, 10
central_params = (V, W, R, a, Zz, R)

In [7]:
so_params = [
    (10, 0, R, a),
    (0, 0, R, a),
]
wvfxns = []
xs = []

for p in so_params:
    wvp, wvm = solver.integral_workspace.wavefunctions(
        sgrid,
        interaction_central,
        interaction_spin_orbit,
        central_params,
        p,
    )
    wvfxns.append((wvp, wvm))

    xs.append(
        solver.xs(
            interaction_central,
            interaction_spin_orbit,
            central_params,
            p,
        )
    )

In [8]:
r = sgrid / kinematics.k

In [12]:
fig, axes = plt.subplots(4, 1, figsize=(6, 8))
axes[0].plot(
    r, interaction_spin_orbit(r, *so_params[0]).real, color=colors[0], alpha=0.7
)
axes[0].plot(
    r, interaction_spin_orbit(r, *so_params[1]).real, color=colors[1], alpha=0.7
)
#axes[0].set_xlabel(r"$r$ [fm]")
axes[0].set_ylabel(r"$U_{so}(r)$ [MeV]")

axes[1].plot(r, wvfxns[0][0][3, :].real, color=colors[0], alpha=0.8)
axes[1].plot(r, wvfxns[1][0][3, :].real, color=colors[1], alpha=0.4)
axes[1].plot(r, wvfxns[0][1][3, :].real, "--", color=colors[0], alpha=0.9)
axes[1].plot(r, wvfxns[1][1][3, :].real, "--", color=colors[1], alpha=0.9)
axes[1].plot([], [], "k", label="$j=7/2$")
axes[1].plot([], [], "--k", label="$j=5/2$")
fig.legend(ncol=2, framealpha=0, bbox_to_anchor=(0.95, 0.77), fontsize=12)
axes[1].set_xlabel(r"$r$ [fm]")
axes[1].set_ylabel(r"Re $ \langle r | f_{j}\rangle $ [a.u.]")

axes[2].semilogy(np.rad2deg(angles), xs[0].dsdo, color=colors[0], alpha=0.7)
axes[2].semilogy(np.rad2deg(angles), xs[1].dsdo, color=colors[1], alpha=0.7)
axes[2].set_xlabel(r"$\theta$ [CM-degrees]")`
axes[2].set_ylabel(r"$d\sigma/d\Omega$ [mb/Sr]")

axes[3].plot(np.rad2deg(angles), xs[0].Ay, color=colors[0], alpha=0.7)
axes[3].plot(np.rad2deg(angles), xs[1].Ay, color=colors[1], alpha=0.7)
axes[3].set_xlabel(r"$\theta$ [CM-degrees]")
axes[3].set_ylabel(r"$A_y$ [unitless]")

plt.tight_layout()
plt.savefig("so.pgf", format="pgf")